# Import & Load Data

In [1]:
import pandas as pd

# Load Data 
df = pd.read_csv("AB_NYC_2019.csv")

In [2]:
print (df.head())

     id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  minimum_nights  number_of_reviews last_review  \
0     Private room    149               1                  9  20

In [3]:
print (df.shape)

(48895, 16)


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     

# Handle Missing Data

In [5]:
# Check missing values
print(df.isnull().sum())  

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


In [6]:
# Drop rows where 'name' or 'host_name' is missing
df.dropna(subset=["name","host_name"], inplace=True)

In [7]:
# Fill missing 'reviews_per_month' with 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

In [8]:
# Convert 'last_review' to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

In [9]:
# Verify again
print(df.isnull().sum())

id                                    0
name                                  0
host_id                               0
host_name                             0
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10037
reviews_per_month                     0
calculated_host_listings_count        0
availability_365                      0
dtype: int64


 # Remove Duplicates

In [10]:
# Check duplicate rows
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


# Standardization

In [11]:
# Standardize string columns
df['name'] = df['name'].str.strip().str.title()
df['host_name'] = df['host_name'].str.strip().str.title()
df['neighbourhood_group'] = df['neighbourhood_group'].str.strip().str.title()
df['neighbourhood'] = df['neighbourhood'].str.strip().str.title()
df['room_type'] = df['room_type'].str.strip().str.title()

# Check data types
print(df.dtypes)

id                                         int64
name                                      object
host_id                                    int64
host_name                                 object
neighbourhood_group                       object
neighbourhood                             object
latitude                                 float64
longitude                                float64
room_type                                 object
price                                      int64
minimum_nights                             int64
number_of_reviews                          int64
last_review                       datetime64[ns]
reviews_per_month                        float64
calculated_host_listings_count             int64
availability_365                           int64
dtype: object


# Outlier Detection & Removal

In [12]:
# Check price outliers
print(df['price'].describe())

count    48858.000000
mean       152.740309
std        240.232386
min          0.000000
25%         69.000000
50%        106.000000
75%        175.000000
max      10000.000000
Name: price, dtype: float64


In [13]:

# Filter price: remove listings above 99th percentile
price_limit = df['price'].quantile(0.99)
df = df[df['price'] < price_limit]

In [14]:
# Remove listings with 0 or extremely high 'minimum_nights'
df = df[(df['minimum_nights'] > 0) & (df['minimum_nights'] < df['minimum_nights'].quantile(0.99))]


In [15]:
# Check again
print(df[['price', 'minimum_nights']].describe())

              price  minimum_nights
count  47874.000000    47874.000000
mean     137.393596        5.703994
std      103.183326        8.371627
min        0.000000        1.000000
25%       69.000000        1.000000
50%      105.000000        2.000000
75%      175.000000        5.000000
max      795.000000       39.000000


# Data Integrity Checks

In [16]:
# Check latitude/longitude within NYC bounds
df = df[(df['latitude'] >= 40.5) & (df['latitude'] <= 41.0)]
df = df[(df['longitude'] >= -74.25) & (df['longitude'] <= -73.7)]

In [17]:
# Check matching between neighbourhood_group and neighbourhood (optional groupwise check)
# Count of unique neighbourhoods per group
print(df.groupby("neighbourhood_group")["neighbourhood"].nunique())

neighbourhood_group
Bronx            48
Brooklyn         47
Manhattan        32
Queens           51
Staten Island    42
Name: neighbourhood, dtype: int64


# Save Cleaned Data

In [18]:
df.to_csv("AB_NYC_2019_Cleaned.csv", index=False)
print("✅ Cleaned dataset saved as 'AB_NYC_2019_Cleaned.csv'")


✅ Cleaned dataset saved as 'AB_NYC_2019_Cleaned.csv'


In [19]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 47873 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              47873 non-null  int64         
 1   name                            47873 non-null  object        
 2   host_id                         47873 non-null  int64         
 3   host_name                       47873 non-null  object        
 4   neighbourhood_group             47873 non-null  object        
 5   neighbourhood                   47873 non-null  object        
 6   latitude                        47873 non-null  float64       
 7   longitude                       47873 non-null  float64       
 8   room_type                       47873 non-null  object        
 9   price                           47873 non-null  int64         
 10  minimum_nights                  47873 non-null  int64         
 11  number_

# Final Compilation Print Block

In [20]:
print("\n📦 DATA CLEANING COMPLETED SUCCESSFULLY")
print("----------------------------------------")
print(f"🔹 Original Rows: 48895")
print(f"🔹 Cleaned Rows : {df.shape[0]}")
print(f"🔹 Columns      : {df.shape[1]}")
print(f"🔹 Missing 'last_review' values: {df['last_review'].isnull().sum()}")
print(f"🔹 Duplicate Rows Removed: {48895 - df.shape[0]}")
print(f"🔹 Cleaned File Saved As: AB_NYC_2019_Cleaned.csv")
print("----------------------------------------")
print("✅ Dataset is now ready for analysis or Power BI visualization.")



📦 DATA CLEANING COMPLETED SUCCESSFULLY
----------------------------------------
🔹 Original Rows: 48895
🔹 Cleaned Rows : 47873
🔹 Columns      : 16
🔹 Missing 'last_review' values: 9593
🔹 Duplicate Rows Removed: 1022
🔹 Cleaned File Saved As: AB_NYC_2019_Cleaned.csv
----------------------------------------
✅ Dataset is now ready for analysis or Power BI visualization.
